# Lab 4 · Entrenamiento en la nube (Colab / Kaggle)

**Todo el código vive en el repo.** Este notebook solo lo trae, lo instala y lanza `python -m training.train`.
Para iterar: edita el código/YAML en tu máquina → `git push` → **Run all** aquí.

- Colab: Runtime → GPU (T4) o CPU. Los runs se guardan en tu Drive (`lab4_runs/`) y **se reanudan solos** si se corta la sesión.
- Kaggle: activa *Internet* en Settings. Los runs quedan en `/kaggle/working/runs` (Save Version → Run All para persistir).

In [ ]:
#@title 1 · Parámetros (lo único que se toca entre corridas)
REPO_URL = "https://github.com/Vasco2510/walker.git"
BRANCH = "main"
CONFIG = "configs/sac_curriculum.yaml"
OVERRIDES = []        # p.ej. ["name=sac_v2", "algo_kwargs.learning_rate=0.0001", "stages.0.timesteps=100000"]
FRESH = False         # True = ignorar checkpoints y empezar de cero
SECRET_NAME = None    # repo privado: nombre del secret con tu token de GitHub (Colab Secrets / Kaggle Add-ons)

In [ ]:
#@title 2 · Bootstrap: clonar/actualizar repo e instalar dependencias
import os, subprocess, sys
from pathlib import Path

def sh(cmd, cwd=None):
    """Ejecuta mostrando la salida en vivo; devuelve las líneas."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line.rstrip())
    if p.wait() != 0:
        raise RuntimeError(f"Falló: {cmd}")
    return lines

ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if Path("training").is_dir() and Path("walker_lab").is_dir():      # ya estamos dentro del repo (uso local)
    REPO = Path.cwd()
else:
    url = REPO_URL
    if SECRET_NAME:
        if ON_COLAB:
            from google.colab import userdata; token = userdata.get(SECRET_NAME)
        else:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret(SECRET_NAME)
        url = url.replace("https://", f"https://{token}@")
    REPO = Path("/kaggle/working/repo" if ON_KAGGLE else "/content/repo")
    if (REPO / ".git").is_dir():
        sh(f"git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD", cwd=REPO)
    else:
        sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")

if ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")   # los runs persisten aquí

os.chdir(REPO)
sh("pip install -q swig && pip install -q -r requirements-train.txt")   # swig por si Box2D debe compilarse
sh("git log -1 --oneline")

In [ ]:
#@title 2b · (Opcional, ~4 min) Calibrar velocidad en ESTA máquina y pegar la salida
sh(f"{sys.executable} -u -m training.benchmark")

In [ ]:
#@title 3 · Entrenar (se reanuda solo si ya hay checkpoint)
args = f"--config {CONFIG} " + " ".join(f"--set {o}" for o in OVERRIDES) + (" --fresh" if FRESH else "")
out = sh(f"{sys.executable} -u -m training.train {args}")
RUN_DIR = Path(next(l for l in reversed(out) if l.startswith("RUN_DIR=")).split("=", 1)[1])

In [ ]:
#@title 4 · Exportar agente, comprobar con evaluar.py oficial y empaquetar
import shutil
sh(f"{sys.executable} -m training.export {RUN_DIR} --to agente")                      # best_agent -> agente/
sh(f"{sys.executable} evaluar.py --scenario escenarios/validacion/maraton_01.json")  # requisito de la entrega
print((RUN_DIR / "best_agent" / "meta.json").read_text()[:700])
zip_path = shutil.make_archive(str(RUN_DIR / "agente"), "zip", ".", "agente")        # el ZIP contiene la carpeta agente/
print("Agente listo:", zip_path)
if ON_COLAB:
    from google.colab import files; files.download(zip_path)
elif ON_KAGGLE:
    shutil.copy(zip_path, "/kaggle/working/agente.zip")   # descargable desde la pestaña Output

In [ ]:
#@title 5 · Reporte realista del run (pegable): velocidad, reparto de tiempo, ETA y evaluaciones
sh(f"{sys.executable} -m training.report {RUN_DIR}")